# Pendulum

In [1]:
import numpy as np
from scipy.integrate import solve_ivp, odeint
import deepdish as dd

def pendulum(t, thetas, g, l):
    theta, dot_theta = thetas # y0, y1
    dots = (dot_theta, -(g/l)*np.sin(theta))
    return  dots # y0_dot, y1_dot

def create_dataset(n_datapoints, timesteps, dt, angles_bound, length_bound, g):
    min_angle, max_angle = angles_bound
    min_len, max_len = length_bound
    
    tmin = 0.0
    tmax = timesteps*dt
    ts = np.linspace(tmin, tmax, timesteps)

    labels = [] # np.empty(n_datapoints) #
    cartesian = np.empty((n_datapoints, timesteps, 2)) # 2d of motion
    phase_space = np.empty((n_datapoints, timesteps, 2)) # 2 degrees of freedom

    for i in range(n_datapoints):
        initial_angle = (max_angle - min_angle) * np.random.random_sample() + min_angle 
        theta0 = np.radians(initial_angle) # initial anglee
        omega0 = 0.0 # initial velocity

        length = (max_len - min_len) * i/(n_datapoints-1) + min_len
        sol = solve_ivp(pendulum, [tmin, tmax], [theta0, omega0], t_eval = ts, args=(g,length))

        # save the x, y coordinated of the pendulum
        xy = np.zeros_like(sol.y)
        xy[0] = length*np.sin(sol.y[0])
        xy[1] = length*np.cos(sol.y[0])
        cartesian[i] = xy.T

        phase_space[i] = sol.y.T

        labels.append({'initial_angle': initial_angle, 
                       'initial_velocity': omega0, 
                       'gravity': g, 
                       'length': length})

        if i % 500 == 0:
            print(i, length, initial_angle)
    dataset = {'cartesian': cartesian, 'phase_space': phase_space, 'labels': labels}
    return dataset

/scratch/tmp/ipykernel_84304/1131370563.py:2: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.23.4)
  from scipy.integrate import solve_ivp, odeint


# Train & In-Dist Test Set

In [2]:
dt = 0.01
timesteps = 2000
angles_bound = (10, 170)
g = 9.81
ang_str = '-'.join([str(a) for a in angles_bound])

In [3]:
print(np.__version__)
print(np.object)
length_bound = (1.0, 1.5)
n_datapoints = 10000
dataset_train = create_dataset(n_datapoints, timesteps, dt, angles_bound, length_bound, g)
len_str = '-'.join([f'{a:.2f}' for a in length_bound])
dd.io.save(f'../data/pendulum_n_{n_datapoints}_steps_{timesteps}_dt_{dt}_len_{len_str}_angle_{ang_str}_g_{g}.hd5', dataset_train)

1.23.4
<class 'object'>
0 1.0 90.3946858793054


/scratch/tmp/ipykernel_84304/4093561357.py:2: DeprecationWarning: `np.object` is a deprecated alias for the builtin `object`. To silence this warning, use `object` by itself. Doing this will not modify any behavior and is safe. 
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  print(np.object)


500 1.025002500250025 159.27919024203985
1000 1.05000500050005 64.80487107293918
1500 1.075007500750075 102.30696930160502
2000 1.1000100010001 157.27612241310987
2500 1.125012501250125 150.78978716310033
3000 1.15001500150015 84.64977859691828
3500 1.175017501750175 94.75118690615064
4000 1.2000200020002 43.83026571466061
4500 1.225022502250225 121.0274819627464
5000 1.25002500250025 77.90166556710378
5500 1.275027502750275 96.16173660439225
6000 1.3000300030003 103.59127976415891
6500 1.325032503250325 127.56528658565279
7000 1.35003500350035 68.61180226891713
7500 1.375037503750375 29.954183210163734
8000 1.4000400040004 109.23783367890496
8500 1.4250425042504251 159.21808873942788
9000 1.45004500450045 49.3026622628938
9500 1.475047504750475 21.914343471585234


# Test set 1

In [4]:
length_bound = (0.90, 1.00)
n_datapoints = 1000
dataset_test1 = create_dataset(n_datapoints, timesteps, dt, angles_bound, length_bound, g)
len_str = '-'.join([f'{a:.2f}' for a in length_bound])
dd.io.save(f'../data/pendulum_n_{n_datapoints}_steps_{timesteps}_dt_{dt}_len_{len_str}_angle_{ang_str}_g_{g}.hd5', dataset_test1)

0 0.9 91.73986284879194
500 0.95005005005005 46.243856104302296


# Test set 2

In [5]:
length_bound = (1.50, 1.60)
n_datapoints = 1000
dataset_test2 = create_dataset(n_datapoints, timesteps, dt, angles_bound, length_bound, g)
len_str = '-'.join([f'{a:.2f}' for a in length_bound])
dd.io.save(f'../data/pendulum_n_{n_datapoints}_steps_{timesteps}_dt_{dt}_len_{len_str}_angle_{ang_str}_g_{g}.hd5', dataset_test2)

0 1.5 44.738042627487346
500 1.55005005005005 168.37030217563074
